In [2]:
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, average_precision_score, precision_score, recall_score

In [3]:
# Cell 2: Load datasets and define key columns
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

outcome_col = "conversion"
treatment_col = "treatment"

feature_cols = [c for c in train_df.columns if c != outcome_col]
X_train = train_df[feature_cols].copy()
y_train = train_df[outcome_col].astype(int)
X_test = test_df[feature_cols].copy()
y_test = test_df[outcome_col].astype(int)

print("Train rows:", len(train_df), "| Test rows:", len(test_df))

Train rows: 800000 | Test rows: 200000


In [4]:
# Cell 3: Train LightGBM S-learner without encoding categoricals
numeric_cols = X_train.select_dtypes(include=[np.number, "bool"]).columns.tolist()
categorical_cols = [c for c in X_train.columns if c not in numeric_cols]

# Use pandas categorical dtype so LightGBM can handle categorical features natively.
for col in categorical_cols:
    combined = pd.concat([X_train[col], X_test[col]], axis=0).astype("string")
    categories = pd.Index(combined.dropna().unique())
    X_train[col] = pd.Categorical(X_train[col].astype("string"), categories=categories)
    X_test[col] = pd.Categorical(X_test[col].astype("string"), categories=categories)

s_learner = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42,
    n_jobs=-1,
)

s_learner.fit(X_train, y_train, categorical_feature=categorical_cols)
print("LightGBM S-learner trained (native categorical handling, no encoding).")

[LightGBM] [Info] Number of positive: 2334, number of negative: 797666
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.025824 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1707
[LightGBM] [Info] Number of data points in the train set: 800000, number of used features: 15
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.002917 -> initscore=-5.834106
[LightGBM] [Info] Start training from score -5.834106
LightGBM S-learner trained (native categorical handling, no encoding).


In [5]:
# Cell 4: Evaluate factual performance on test.csv
factual_proba = s_learner.predict_proba(X_test)[:, 1]
factual_pred = (factual_proba >= 0.5).astype(int)

precision = precision_score(y_test, factual_pred, zero_division=0)
recall = recall_score(y_test, factual_pred, zero_division=0)
accuracy = accuracy_score(y_test, factual_pred)
auprc = average_precision_score(y_test, factual_proba)

print("Test metrics on test.csv")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"Accuracy:  {accuracy:.4f}")
print(f"AUPRC:     {auprc:.4f}")

Test metrics on test.csv
Precision: 0.4976
Recall:    0.1750
Accuracy:  0.9971
AUPRC:     0.3187


In [6]:
# Cell 5: Counterfactual conversion predictions with treatment toggled
X_test_treated = X_test.copy()
X_test_control = X_test.copy()
X_test_treated[treatment_col] = 1
X_test_control[treatment_col] = 0

mu1 = s_learner.predict_proba(X_test_treated)[:, 1]  # P(conversion | do(treatment=1), x)
mu0 = s_learner.predict_proba(X_test_control)[:, 1]  # P(conversion | do(treatment=0), x)
uplift = mu1 - mu0

results = test_df.copy()
results["p_conversion_treated"] = mu1
results["p_conversion_control"] = mu0
results["uplift"] = uplift
results

,f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,f11,treatment,conversion,visit,exposure,p_conversion_treated,p_conversion_control,uplift
0,23.032518,10.059654,8.214383,4.679882,10.280525,4.115453,-12.422866,4.833815,3.971858,13.190056,5.300375,-0.168679,1,0,0,0,4.495893e-07,4.918823e-07,-4.229292e-08
1,25.600311,10.059654,8.214383,4.679882,10.280525,4.115453,-1.288207,4.833815,3.971858,13.190056,5.300375,-0.168679,1,0,0,0,4.075863e-07,4.152930e-07,-7.706656e-09
2,19.523195,10.059654,8.639891,3.359763,10.280525,4.115453,-7.301017,4.833815,3.878372,25.240993,5.300375,-0.168679,1,0,0,0,5.875868e-07,5.986969e-07,-1.111011e-08
3,12.616365,10.059654,8.422836,4.679882,10.280525,4.115453,0.294443,4.833815,3.835851,18.380112,5.300375,-0.168679,1,0,0,0,5.219299e-07,5.534747e-07,-3.154479e-08
4,20.454730,10.059654,8.847605,3.907662,10.280525,4.115453,-10.006574,4.833815,3.899112,13.190056,5.300375,-0.168679,1,0,0,0,6.145677e-07,6.261880e-07,-1.162027e-08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199995,12.616365,10.059654,9.040947,4.679882,10.280525,4.115453,0.294443,4.833815,3.943716,13.190056,5.300375,-0.168679,1,0,0,0,2.926512e-07,2.981847e-07,-5.533460e-09
199996,24.811454,10.059654,8.214383,4.679882,10.280525,4.115453,-1.288207,4.833815,3.971858,13.190056,5.300375,-0.168679,0,0,0,0,4.075863e-07,4.152930e-07,-7.706656e-09
199997,22.928756,10.059654,8.378748,4.679882,10.280525,3.013064,-15.877431,11.560131,3.771810,46.672202,5.300375,-0.168679,0,0,0,0,5.261753e-07,5.316364e-07,-5.461095e-09
199998,12.616365,10.059654,8.841462,4.679882,10.280525,4.115453,0.294443,4.833815,3.943716,13.190056,5.300375,-0.168679,1,0,0,0,5.719119e-07,5.827256e-07,-1.081373e-08


In [8]:
# Cell 6: Save trained S-learner model to PKL
import pickle

s_learner_bundle = {
    "model": s_learner,
    "feature_cols": feature_cols,
    "categorical_cols": categorical_cols,
    "treatment_col": treatment_col,
    "outcome_col": outcome_col,
}

with open("s_learner.pkl", "wb") as f:
    pickle.dump(s_learner_bundle, f)

print("Saved s_learner.pkl")

Saved s_learner.pkl
